In [1]:
from transformers import AutoModelForMaskedLM

In [ ]:
model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint, token=False)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [3]:
parameters = model.num_parameters() / 1_000_000
print(f"The model has {parameters:.2f} million parameters")

The model has 66.99 million parameters


In [6]:
text = "This is a great [MASK]."

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint,token=False)


In [11]:
inputs = tokenizer(text, return_tensors="pt")
token_ids = inputs["input_ids"][0]
print(token_ids)
token_strings = tokenizer.convert_ids_to_tokens(token_ids)
print(token_strings)    
decoded_text = tokenizer.decode(inputs["input_ids"][0])
print(decoded_text)


tensor([ 101, 2023, 2003, 1037, 2307,  103, 1012,  102])
['[CLS]', 'this', 'is', 'a', 'great', '[MASK]', '.', '[SEP]']
[CLS] this is a great [MASK]. [SEP]


In [12]:
import torch
inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits

mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
print(mask_token_logits.shape)

top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    word = tokenizer.decode([token])
    new_sentence = text.replace(tokenizer.mask_token, word)
    print(new_sentence)

torch.Size([1, 30522])
This is a great deal.
This is a great success.
This is a great adventure.
This is a great idea.
This is a great feat.


In [13]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb")

Using the latest cached version of the dataset since stanfordnlp/imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\khale\.cache\huggingface\datasets\stanfordnlp___imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Thu Jul 16 18:37:28 2026).


In [ ]:
size_in_mb = dataset["train"].dataset_size / (1024 * 1024)
print(size_in_mb)


127.03209114074707
25000


In [14]:
sample = dataset["train"].shuffle(seed=45).select(range(5))
for row in sample:
    print(row["text"])
    print(f"Label: {row['label']}")


Oh dear, Oh dear. I started watching this not knowing what to expect. I couldn't believe what I was seeing. There were times when I thought it was a comedy. I loved how the government's plan to capture the terrorist leader is to air drop in one man, who is unarmed, and expect him to capture him and escape with a rocket pack. If only it were really that easy. I've finally found a movie worse than "Plan 9 From Outer Space".
Label: 0
Do you guys wanna know a secret?. This movie sucks. Well actually i don't know because if you allow yourself to be indulged by plagiarised versions of original movies, then perhaps you may find this movie astounding (this movie being a plagiarised copy of i know what you did last summer). The first 30 minutes of the movie is based on a typical story setting; a bunch of so-called cool teenagers relishing their vacation in Florida and being themselves by behaving very much like the juveniles they are. The only insight we get at this point is the extent to which

In [15]:
def tokenize_function(examples):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text", "label"]   )

In [16]:
tokenizer.model_max_length

512

In [17]:
chunk_size = 128

In [18]:
# Slicing produces a list of lists for each feature
tokenized_samples = tokenized_datasets["train"][:3]

for idx, sample in enumerate(tokenized_samples["input_ids"]):
    print(f"'Review {idx} length: {len(sample)}'")

'Review 0 length: 363'
'Review 1 length: 304'
'Review 2 length: 133'


In [19]:
concatenated_examples = {
    k: sum(tokenized_samples[k], []) for k in tokenized_samples.keys()
}
total_length = len(concatenated_examples["input_ids"])
print(f"'Concatenated reviews length: {total_length}'")

'Concatenated reviews length: 800'


In [20]:
chunks = {
    k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
    for k, t in concatenated_examples.items()
}

for chunk in chunks["input_ids"]:
    print(f"'>>> Chunk length: {len(chunk)}'")

print(chunks)

'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 128'
'>>> Chunk length: 32'
{'input_ids': [[101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009, 2043, 2009, 2001, 2034, 2207, 1999, 3476, 1012, 1045, 2036, 2657, 2008, 2012, 2034, 2009, 2001, 8243, 2011, 1057, 1012, 1055, 1012, 8205, 2065, 2009, 2412, 2699, 2000, 4607, 2023, 2406, 1010, 3568, 2108, 1037, 5470, 1997, 3152, 2641, 1000, 6801, 1000, 1045, 2428, 2018, 2000, 2156, 2023, 2005, 2870, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 5436, 2003, 8857, 2105, 1037, 2402, 4467, 3689, 3076, 2315, 14229, 2040, 4122, 2000, 4553, 2673, 2016, 2064, 2055, 2166, 1012, 1999, 3327, 2016, 4122, 2000, 3579, 2014, 3086, 2015, 2000, 2437, 2070, 4066, 1997, 4516, 2006, 2054, 1996, 2779, 25430, 14728, 2245, 2055, 3056, 2576, 3314, 2107], [2004, 1996, 5148, 2162, 1998, 2679, 3314, 19

In [21]:
def group_texts(examples):
    # Concatenate all texts
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}

    total_length = len(concatenated_examples[list(examples.keys())[0]])

    total_length = (total_length // chunk_size) * chunk_size

    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    
    result["labels"] = result["input_ids"].copy()
    return result

In [22]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)
lm_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 122957
    })
})

In [23]:
tokenizer.decode(lm_datasets["train"][0]["input_ids"])

'[CLS] i rented i am curious - yellow from my video store because of all the controversy that surrounded it when it was first released in 1967. i also heard that at first it was seized by u. s. customs if it ever tried to enter this country, therefore being a fan of films considered " controversial " i really had to see this for myself. < br / > < br / > the plot is centered around a young swedish drama student named lena who wants to learn everything she can about life. in particular she wants to focus her attentions to making some sort of documentary on what the average swede thought about certain political issues such'

In [24]:
tokenizer.decode(lm_datasets["train"][0]["labels"])

'[CLS] i rented i am curious - yellow from my video store because of all the controversy that surrounded it when it was first released in 1967. i also heard that at first it was seized by u. s. customs if it ever tried to enter this country, therefore being a fan of films considered " controversial " i really had to see this for myself. < br / > < br / > the plot is centered around a young swedish drama student named lena who wants to learn everything she can about life. in particular she wants to focus her attentions to making some sort of documentary on what the average swede thought about certain political issues such'

In [25]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [26]:
samples = [lm_datasets["train"][i] for i in range(3)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'{tokenizer.decode(chunk)}'")


'[CLS] i rented i am curious - yellow from my video pioneering because of all the controversy that surrounded it when it [MASK] first released in 1967. i [MASK] heard that at first [MASK] was seized by u [MASK] [MASK] [MASK] [MASK] if it ever tried to enter this country, therefore being a fan of films considered " controversial " i really had [unused485] see this for [MASK]. < br / > < br / > the plot is centered around a [MASK] swedish drama student named lena who wants to learn everything she can about life. in particular she [MASK] to focus her attentions vol making some sort of documentary on what the average progressingede thought about certain political issues such'

'as the vietnam [MASK] and race issues in the united [MASK]. in [MASK] asking politicians and ordinary denizens [MASK] [MASK] about their opinions on politics derives [MASK] has sex with her drama [MASK], classmates, and married men. < br / > [MASK] br / > what kills me about i am curious - yellow [MASK] that 40 yea

In [ ]:
import collections
import numpy as np

from transformers import default_data_collator

wwm_probability = 0.2


def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # Create a map between words and corresponding token indices
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)

        # Randomly mask words
        mask = np.random.binomial(1, wwm_probability, (len(mapping),))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels

    return default_data_collator(features)

In [44]:
samples = [lm_datasets["train"][i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)

for chunk in batch["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] i rented i am curious - [MASK] from [MASK] video store [MASK] of all [MASK] controversy that surrounded it when it was [MASK] released in 1967. i also heard that at first [MASK] was seized by u. s. customs [MASK] it ever tried to enter this country, [MASK] being [MASK] fan [MASK] films considered " controversial " i really had [MASK] see this [MASK] myself. < br / [MASK] < [MASK] / > the plot [MASK] [MASK] [MASK] a young swedish drama [MASK] named [MASK] who [MASK] [MASK] learn everything she can about life. in particular she wants to [MASK] her [MASK] [MASK] to making some sort [MASK] documentary on what the average swede thought about certain [MASK] issues such'

'>>> as [MASK] vietnam war and [MASK] issues [MASK] the [MASK] [MASK]. in between [MASK] politicians and ordinary [MASK] [MASK] [MASK] [MASK] stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married [MASK] [MASK] < br [MASK] > < br [MASK] > what kills [MASK] [MASK] i

In [45]:
train_size = 10_000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_datasets["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
downsampled_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 1000
    })
})

In [46]:
from huggingface_hub import notebook_login

notebook_login()

In [68]:
from transformers import TrainingArguments

batch_size = 64
# Show the training loss with every epoch
logging_steps = len(downsampled_dataset["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-imdb",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps,
)

In [71]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [72]:
import math

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
No log,2.467891,0


>>> Perplexity: 11.80


In [73]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.557359,2.450104
2,2.541129,2.421986
3,2.491240,2.456576


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=471, training_loss=2.529690287928166, metrics={'train_runtime': 1038.3803, 'train_samples_per_second': 28.891, 'train_steps_per_second': 0.454, 'total_flos': 994208670720000.0, 'train_loss': 2.529690287928166, 'epoch': 3.0})

In [76]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
2.491240,2.429128,3


>>> Perplexity: 11.35


In [77]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/KhaledAalnobani/distilbert-base-uncased-finetuned-imdb/commit/129c55bb40f1561266e48983fa79ee445fe190eb', commit_message='End of training', commit_description='', oid='129c55bb40f1561266e48983fa79ee445fe190eb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/KhaledAalnobani/distilbert-base-uncased-finetuned-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='KhaledAalnobani/distilbert-base-uncased-finetuned-imdb'), pr_revision=None, pr_num=None)

In [78]:
def insert_random_mask(batch):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    masked_inputs = data_collator(features)
    # Create a new "masked" column for each column in the dataset
    return {"masked_" + k: v.numpy() for k, v in masked_inputs.items()}

In [79]:
downsampled_dataset = downsampled_dataset.remove_columns(["word_ids"])
eval_dataset = downsampled_dataset["test"].map(
    insert_random_mask,
    batched=True,
    remove_columns=downsampled_dataset["test"].column_names,
)
eval_dataset = eval_dataset.rename_columns(
    {
        "masked_input_ids": "input_ids",
        "masked_attention_mask": "attention_mask",
        "masked_labels": "labels",
    }
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [81]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

batch_size = 64
train_dataloader = DataLoader(
    downsampled_dataset["train"],
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)
eval_dataloader = DataLoader(
    eval_dataset, batch_size=batch_size, collate_fn=default_data_collator
)

In [82]:
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [85]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr =5e-5)

In [86]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [87]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [92]:
from huggingface_hub import create_repo, get_full_repo_name

model_name = "distilbert-base-uncased-finetuned-imdb-accelerate"
output_dir = model_name

repo_name = get_full_repo_name(model_name)

create_repo(repo_name, exist_ok=True)

print(repo_name)

KhaledAalnobani/distilbert-base-uncased-finetuned-imdb-accelerate


In [93]:
from tqdm.auto import tqdm
import math
import torch


progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    model.train()
    for batch in train_dataloader:
        output = model(**batch)
        loss = output.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            outputs = model(**batch)

        loss = outputs.loss
        losses.append(accelerator.gather(loss.repeat(batch_size)))

    losses = torch.cat(losses)
    losses = losses[: len(eval_dataset)] 
    
    try:
        perplexity = math.exp(torch.mean(losses))
    except OverflowError:
        perplexity = float("inf")

    print(f">>> Epoch {epoch}: Perplexity: {perplexity}")

    accelerator.wait_for_everyone()
    
    unwrapped_model = accelerator.unwrap_model(model)
    
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        
        unwrapped_model.push_to_hub(
            repo_name, 
            commit_message=f"Training in progress epoch {epoch}"
        )
        tokenizer.push_to_hub(
            repo_name, 
            commit_message=f"Tokenizer for epoch {epoch}"
        )
        

  0%|          | 0/471 [00:00<?, ?it/s]

>>> Epoch 0: Perplexity: 11.217967213063332


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


In [ ]:
from transformers import pipeline

mask_filler = pipeline(
    "fill-mask", 
    model=repo_name, 
    tokenizer=repo_name 
)

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

In [95]:
preds = mask_filler(text)

for pred in preds:
    print(f">>> {pred['sequence']}")

>>> this is a great film.
>>> this is a great movie.
>>> this is a great idea.
>>> this is a great story.
>>> this is a great adventure.
